# Session 14: Network Troubleshooting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-14-network-troubleshooting.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-14_troubleshoot.py`

In [1]:
"""Session 14 - The troubleshooting methodology as code: a bottom-up checker
that runs only SAFE, LOCAL checks on your own machine, plus a symptom ->
likely-cause lookup that follows the OSI layers.

Run:  python session-14_troubleshoot.py
"""
import socket
import ipaddress

SYMPTOMS = {
    "no link light":              (1, "cable unplugged/damaged, wrong port, NIC disabled"),
    "link up but 169.254.x.x ip": (3, "DHCP server unreachable: check VLAN, DHCP scope, relay"),
    "can ping ip but not name":   (7, "DNS server wrong or down: check resolver settings"),
    "slow and many crc errors":   (1, "bad cable, EMI, duplex mismatch"),
    "one vlan cannot reach others": (3, "missing inter-VLAN route or trunk not allowing VLAN"),
    "intermittent drop on wi-fi": (1, "interference, channel overlap, weak signal"),
    "website loads but not https": (4, "firewall blocks 443 or certificate problem"),
}


def step(name, ok, detail):
    print(f"  [{'OK ' if ok else 'BAD'}] {name:<34} {detail}")
    return ok


def local_checks():
    print("Bottom-up checks on this machine:")
    host = socket.gethostname()
    step("Host name", bool(host), host)
    try:
        lo = socket.gethostbyname("localhost")
        step("Loopback resolves (TCP/IP stack)", ipaddress.ip_address(lo).is_loopback, lo)
    except OSError as e:
        step("Loopback resolves (TCP/IP stack)", False, str(e))
    srv = socket.socket(); srv.bind(("127.0.0.1", 0)); srv.listen(1)
    port = srv.getsockname()[1]
    try:
        socket.create_connection(("127.0.0.1", port), timeout=1).close()
        step("TCP connect to local service", True, "handshake completed on 127.0.0.1")
    except OSError as e:
        step("TCP connect to local service", False, str(e))
    srv.close()
    try:
        socket.create_connection(("127.0.0.1", port), timeout=1).close()
        step("Same port after service stopped", True, "unexpected")
    except OSError:
        step("Same port after service stopped", False, "connection refused = nothing listening (Layer 4/7)")


def diagnose(symptom):
    layer, cause = SYMPTOMS[symptom]
    return f"Layer {layer}: {cause}"


if __name__ == "__main__":
    local_checks()
    print("\nSymptom -> where to start:")
    for s in SYMPTOMS:
        print(f"  {s:<30} {diagnose(s)}")

Bottom-up checks on this machine:
  [OK ] Host name                          (none)
  [OK ] Loopback resolves (TCP/IP stack)   127.0.0.1
  [OK ] TCP connect to local service       handshake completed on 127.0.0.1
  [BAD] Same port after service stopped    connection refused = nothing listening (Layer 4/7)

Symptom -> where to start:
  no link light                  Layer 1: cable unplugged/damaged, wrong port, NIC disabled
  link up but 169.254.x.x ip     Layer 3: DHCP server unreachable: check VLAN, DHCP scope, relay
  can ping ip but not name       Layer 7: DNS server wrong or down: check resolver settings
  slow and many crc errors       Layer 1: bad cable, EMI, duplex mismatch
  one vlan cannot reach others   Layer 3: missing inter-VLAN route or trunk not allowing VLAN
  intermittent drop on wi-fi     Layer 1: interference, channel overlap, weak signal
  website loads but not https    Layer 4: firewall blocks 443 or certificate problem


## `mini_project_netplan.py`

In [2]:
"""Module 3 mini project - "Campus Branch Network Planner".

Given a branch office's departments and head-counts, the planner:
  1. allocates VLSM subnets from one private block (largest first, no overlap),
  2. assigns VLAN IDs and gateway addresses,
  3. estimates switch ports and PoE access points,
  4. writes an inter-VLAN ACL policy (guest isolated, servers protected),
  5. prints a documentation table (the deliverable a network admin hands over).
Run:  python mini_project_netplan.py
"""
import ipaddress
import math

BLOCK = ipaddress.ip_network("10.20.0.0/22")
DEPARTMENTS = [  # name, hosts needed today, growth %, wired?
    ("Staff",        110, 20, True),
    ("Students-Lab",  60, 10, True),
    ("Guest-WiFi",    90, 30, False),
    ("Servers",       12, 50, True),
    ("Management",     6, 50, True),
]
P2P_LINKS = 2          # router-to-firewall and router-to-ISP style links (/30)


def needed_prefix(hosts):
    """Smallest prefix with enough usable addresses (hosts + gateway)."""
    return 32 - math.ceil(math.log2(hosts + 1 + 2))


def allocate():
    plan, free = [], [BLOCK]
    reqs = []
    for name, h, growth, wired in DEPARTMENTS:
        future = math.ceil(h * (1 + growth / 100))
        reqs.append((name, h, future, wired))
    reqs.sort(key=lambda r: -r[2])                       # VLSM: biggest first
    reqs += [(f"P2P-{i + 1}", 2, 2, True) for i in range(P2P_LINKS)]
    for i, (name, h, future, wired) in enumerate(reqs):
        prefix = 30 if name.startswith("P2P") else needed_prefix(future)
        free.sort(key=lambda n: (n.network_address))
        for blk in free:
            if blk.prefixlen <= prefix:
                sub = next(blk.subnets(new_prefix=prefix))
                free.remove(blk)
                free += list(blk.address_exclude(sub))
                plan.append((name, h, future, wired, sub))
                break
        else:
            raise SystemExit(f"Block {BLOCK} is too small for {name}")
    return plan, free


if __name__ == "__main__":
    plan, free = allocate()
    print(f"Address block {BLOCK} ({BLOCK.num_addresses} addresses)\n")
    print(f"{'VLAN':<5}{'Segment':<14}{'need':>5}{'+grow':>6}  {'subnet':<17}{'gateway':<14}{'usable':>7}{'spare':>6}")
    vlan = 10
    for name, h, future, wired, sub in plan:
        usable = sub.num_addresses - 2
        gw = next(sub.hosts())
        vid = "-" if name.startswith("P2P") else str(vlan)
        print(f"{vid:<5}{name:<14}{h:>5}{future:>6}  {str(sub):<17}{str(gw):<14}{usable:>7}{usable - future - (0 if name.startswith('P2P') else 1):>6}")
        if not name.startswith("P2P"):
            vlan += 10
    used = sum(s.num_addresses for *_, s in plan)
    print(f"\nUsed {used}/{BLOCK.num_addresses} addresses ({used / BLOCK.num_addresses:.0%}); "
          f"free blocks: {', '.join(str(f) for f in sorted(free, key=lambda n: n.network_address))}")

    wired_hosts = sum(f for n, h, f, w, s in plan if w and not n.startswith("P2P"))
    ports = wired_hosts + 4                       # + uplinks/APs/printers margin
    switches = math.ceil(ports / 48)
    aps = math.ceil(sum(f for n, h, f, w, s in plan if not w) / 30)   # ~30 clients per AP
    print(f"\nHardware: {wired_hosts} wired devices -> {switches} x 48-port switches; "
          f"{aps} PoE access points for wireless clients")

    subnets = {n: s for n, *_, s in plan}
    print("\nInter-VLAN ACL policy (applied on the router, first match wins):")
    rules = [
        ("deny", "Guest-WiFi", "any internal", "guests reach the internet only"),
        ("permit", "Management", "any", "admins manage everything"),
        ("permit", "Staff", "Servers tcp 443,445", "staff use file and web apps"),
        ("permit", "Students-Lab", "Servers tcp 443", "lab uses the web portal only"),
        ("deny", "any", "Management", "nobody else touches device management"),
        ("permit", "any", "internet", "everything else may browse"),
    ]
    for i, (act, src, dst, why) in enumerate(rules, 1):
        s = subnets.get(src, src)
        print(f"  {i}. {act:<6} {src:<12} ({str(s):<15}) -> {dst:<22} # {why}")

Address block 10.20.0.0/22 (1024 addresses)

VLAN Segment        need +grow  subnet           gateway        usable spare
10   Staff           110   132  10.20.0.0/24     10.20.0.1         254   121
20   Guest-WiFi       90   117  10.20.1.0/25     10.20.1.1         126     8
30   Students-Lab     60    66  10.20.1.128/25   10.20.1.129       126    59
40   Servers          12    18  10.20.2.0/27     10.20.2.1          30    11
50   Management        6     9  10.20.2.32/28    10.20.2.33         14     4
-    P2P-1             2     2  10.20.2.48/30    10.20.2.49          2     0
-    P2P-2             2     2  10.20.2.52/30    10.20.2.53          2     0

Used 568/1024 addresses (55%); free blocks: 10.20.2.56/29, 10.20.2.64/26, 10.20.2.128/25, 10.20.3.0/24

Hardware: 225 wired devices -> 5 x 48-port switches; 4 PoE access points for wireless clients

Inter-VLAN ACL policy (applied on the router, first match wins):
  1. deny   Guest-WiFi   (10.20.1.0/25   ) -> any internal           # gue